In [8]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import polars.selectors as cs
import pandas as pd
import plotly.express as px

import ipaddress
import datetime as dt

import polars.selectors as cs

In [9]:
df_path = r'/mnt/samsung/Datasets/csv_data/Electric_Vehicle_Population_Data.csv'

In [10]:
df = pl.scan_csv(df_path, include_file_paths='path').collect()

In [11]:
df.collect_schema()

Schema([('VIN (1-10)', String),
        ('County', String),
        ('City', String),
        ('State', String),
        ('Postal Code', Int64),
        ('Model Year', Int64),
        ('Make', String),
        ('Model', String),
        ('Electric Vehicle Type', String),
        ('Clean Alternative Fuel Vehicle (CAFV) Eligibility', String),
        ('Electric Range', Int64),
        ('Legislative District', Int64),
        ('DOL Vehicle ID', Int64),
        ('Vehicle Location', String),
        ('Electric Utility', String),
        ('2020 Census Tract', Int64),
        ('path', String)])

In [12]:
df.estimated_size('mb')

82.49908638000488

In [13]:
df.describe()

statistic,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 Census Tract,path
str,str,str,str,str,f64,f64,str,str,str,str,f64,f64,f64,str,str,f64,str
"""count""","""280833""","""280821""","""280821""","""280833""",280821.0,280833.0,"""280833""","""280833""","""280833""","""280833""",280821.0,280130.0,280833.0,"""280813""","""280821""",280821.0,"""280833"""
"""null_count""","""0""","""12""","""12""","""0""",12.0,0.0,"""0""","""0""","""0""","""0""",12.0,703.0,0.0,"""20""","""12""",12.0,"""0"""
"""mean""",null,null,null,null,98175.409538,2022.111643,null,null,null,null,38.757846,28.819352,2.4706e8,null,null,5.2970e10,null
"""std""",null,null,null,null,2591.070456,3.064528,null,null,null,null,77.903581,14.90741,6.3253e7,null,null,1.6548e9,null
"""min""","""1C4JJXN60P""","""Ada""","""Aberdeen""","""AB""",1030.0,1999.0,"""ACURA""","""330E""","""Battery Electric Vehicle (BEV)""","""Clean Alternative Fuel Vehicle…",0.0,1.0,4385.0,"""POINT (-100.48872 31.42094)""","""AVISTA CORP""",1.0010e9,"""/mnt/samsung/Datasets/csv_data…"
"""25%""",null,null,null,null,98052.0,2021.0,null,null,null,null,0.0,17.0,2.24072999e8,null,null,5.3033e10,null
"""50%""",null,null,null,null,98133.0,2023.0,null,null,null,null,0.0,32.0,2.64239482e8,null,null,5.3033e10,null
"""75%""",null,null,null,null,98382.0,2024.0,null,null,null,null,32.0,42.0,2.80300569e8,null,null,5.3054e10,null
"""max""","""ZPBUD6ZLXS""","""Yuma""","""Zillah""","""WY""",99517.0,2027.0,"""WHEEGO ELECTRIC CARS""","""ZEVO""","""Plug-in Hybrid Electric Vehicl…","""Not eligible due to low batter…",337.0,49.0,4.79114996e8,"""POINT (144.9083 13.55991)""","""PUGET SOUND ENERGY INC||PUD NO…",6.6011e10,"""/mnt/samsung/Datasets/csv_data…"


In [14]:
df.select(
    pl.all().n_unique()
)

VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 Census Tract,path
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
17281,251,906,52,1131,23,47,187,2,3,114,50,280833,1130,78,2394,1


In [15]:
df = df.with_columns(
    pl.col('County').cast(pl.Categorical),
    pl.col('City').cast(pl.Categorical),
    pl.col('Postal Code').cast(pl.UInt32),
    pl.col('Model Year').cast(pl.UInt16),
    pl.col('Make').cast(pl.Categorical),
    pl.col('Model').cast(pl.Categorical),
    pl.col('Electric Vehicle Type').cast(pl.Categorical),
    pl.col('Clean Alternative Fuel Vehicle (CAFV) Eligibility').cast(pl.Categorical),
    pl.col('Electric Range').cast(pl.Int16),
    pl.col('Legislative District').cast(pl.Int8),
    pl.col('DOL Vehicle ID').cast(pl.Int32),
    pl.col('path').cast(pl.Categorical),
)

In [16]:
df = df.select(
    pl.all().name.map(lambda word: word.lower().replace(' ', '_'))
)

In [17]:
df.collect_schema()

Schema([('vin_(1-10)', String),
        ('county', Categorical),
        ('city', Categorical),
        ('state', String),
        ('postal_code', UInt32),
        ('model_year', UInt16),
        ('make', Categorical),
        ('model', Categorical),
        ('electric_vehicle_type', Categorical),
        ('clean_alternative_fuel_vehicle_(cafv)_eligibility', Categorical),
        ('electric_range', Int16),
        ('legislative_district', Int8),
        ('dol_vehicle_id', Int32),
        ('vehicle_location', String),
        ('electric_utility', String),
        ('2020_census_tract', Int64),
        ('path', Categorical)])

In [20]:
df

vin_(1-10),county,city,state,postal_code,model_year,make,model,electric_vehicle_type,clean_alternative_fuel_vehicle_(cafv)_eligibility,electric_range,legislative_district,dol_vehicle_id,vehicle_location,electric_utility,2020_census_tract,path
str,cat,cat,str,u32,u16,cat,cat,cat,cat,i16,i8,i32,str,str,i64,cat
"""1N4AZ0CP6D""","""King""","""Kirkland""","""WA""",98034,2013,"""NISSAN""","""LEAF""","""Battery Electric Vehicle (BEV)""","""Clean Alternative Fuel Vehicle…",75,1,154635729,"""POINT (-122.22901 47.72201)""","""PUGET SOUND ENERGY INC||CITY O…",53033022203,"""/mnt/samsung/Datasets/csv_data…"
"""5YJ3E1EC8L""","""Kitsap""","""Bainbridge Island""","""WA""",98110,2020,"""TESLA""","""MODEL 3""","""Battery Electric Vehicle (BEV)""","""Clean Alternative Fuel Vehicle…",308,23,107518645,"""POINT (-122.521 47.62759)""","""PUGET SOUND ENERGY INC""",53035090902,"""/mnt/samsung/Datasets/csv_data…"
"""5YJ3E1EBXJ""","""King""","""Seattle""","""WA""",98144,2018,"""TESLA""","""MODEL 3""","""Battery Electric Vehicle (BEV)""","""Clean Alternative Fuel Vehicle…",215,37,474808813,"""POINT (-122.30866 47.57874)""","""CITY OF SEATTLE - (WA)|CITY OF…",53033009000,"""/mnt/samsung/Datasets/csv_data…"
"""ZFAFFAC45R""","""Thurston""","""Yelm""","""WA""",98597,2024,"""FIAT""","""500E""","""Battery Electric Vehicle (BEV)""","""Eligibility unknown as battery…",0,2,273658514,"""POINT (-122.60735 46.94239)""","""PUGET SOUND ENERGY INC""",53067012412,"""/mnt/samsung/Datasets/csv_data…"
"""5YJYGDEE3L""","""King""","""Kent""","""WA""",98030,2020,"""TESLA""","""MODEL Y""","""Battery Electric Vehicle (BEV)""","""Clean Alternative Fuel Vehicle…",291,47,109579900,"""POINT (-122.19975 47.37483)""","""PUGET SOUND ENERGY INC||CITY O…",53033029507,"""/mnt/samsung/Datasets/csv_data…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""1FT6W1BM2S""","""Whatcom""","""Lynden""","""WA""",98264,2025,"""FORD""","""F-150""","""Battery Electric Vehicle (BEV)""","""Eligibility unknown as battery…",0,42,280662399,"""POINT (-122.45051 48.94298)""","""PUGET SOUND ENERGY INC||PUD NO…",53073010701,"""/mnt/samsung/Datasets/csv_data…"
"""5YJ3E1EB6N""","""Snohomish""","""Sultan""","""WA""",98294,2022,"""TESLA""","""MODEL 3""","""Battery Electric Vehicle (BEV)""","""Eligibility unknown as battery…",0,39,185982247,"""POINT (-121.81688 47.8623)""","""PUGET SOUND ENERGY INC""",53061053803,"""/mnt/samsung/Datasets/csv_data…"
"""JTJBCACB3T""","""Snohomish""","""Lake Stevens""","""WA""",98258,2026,"""LEXUS""","""RZ""","""Battery Electric Vehicle (BEV)""","""Eligibility unknown as battery…",0,39,290127582,"""POINT (-122.06402 48.01497)""","""PUGET SOUND ENERGY INC""",53061053505,"""/mnt/samsung/Datasets/csv_data…"


In [23]:
df.with_columns(
    avernage_year = pl.col('model_year').mean(),
    avernage_year_1 = pl.col('model_year').mean().over('make'),
)

vin_(1-10),county,city,state,postal_code,model_year,make,model,electric_vehicle_type,clean_alternative_fuel_vehicle_(cafv)_eligibility,electric_range,legislative_district,dol_vehicle_id,vehicle_location,electric_utility,2020_census_tract,path,avernage_year,avernage_year_1
str,cat,cat,str,u32,u16,cat,cat,cat,cat,i16,i8,i32,str,str,i64,cat,f64,f64
"""1N4AZ0CP6D""","""King""","""Kirkland""","""WA""",98034,2013,"""NISSAN""","""LEAF""","""Battery Electric Vehicle (BEV)""","""Clean Alternative Fuel Vehicle…",75,1,154635729,"""POINT (-122.22901 47.72201)""","""PUGET SOUND ENERGY INC||CITY O…",53033022203,"""/mnt/samsung/Datasets/csv_data…",2022.111643,2018.785427
"""5YJ3E1EC8L""","""Kitsap""","""Bainbridge Island""","""WA""",98110,2020,"""TESLA""","""MODEL 3""","""Battery Electric Vehicle (BEV)""","""Clean Alternative Fuel Vehicle…",308,23,107518645,"""POINT (-122.521 47.62759)""","""PUGET SOUND ENERGY INC""",53035090902,"""/mnt/samsung/Datasets/csv_data…",2022.111643,2022.311104
"""5YJ3E1EBXJ""","""King""","""Seattle""","""WA""",98144,2018,"""TESLA""","""MODEL 3""","""Battery Electric Vehicle (BEV)""","""Clean Alternative Fuel Vehicle…",215,37,474808813,"""POINT (-122.30866 47.57874)""","""CITY OF SEATTLE - (WA)|CITY OF…",53033009000,"""/mnt/samsung/Datasets/csv_data…",2022.111643,2022.311104
"""ZFAFFAC45R""","""Thurston""","""Yelm""","""WA""",98597,2024,"""FIAT""","""500E""","""Battery Electric Vehicle (BEV)""","""Eligibility unknown as battery…",0,2,273658514,"""POINT (-122.60735 46.94239)""","""PUGET SOUND ENERGY INC""",53067012412,"""/mnt/samsung/Datasets/csv_data…",2022.111643,2016.711924
"""5YJYGDEE3L""","""King""","""Kent""","""WA""",98030,2020,"""TESLA""","""MODEL Y""","""Battery Electric Vehicle (BEV)""","""Clean Alternative Fuel Vehicle…",291,47,109579900,"""POINT (-122.19975 47.37483)""","""PUGET SOUND ENERGY INC||CITY O…",53033029507,"""/mnt/samsung/Datasets/csv_data…",2022.111643,2022.311104
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""1FT6W1BM2S""","""Whatcom""","""Lynden""","""WA""",98264,2025,"""FORD""","""F-150""","""Battery Electric Vehicle (BEV)""","""Eligibility unknown as battery…",0,42,280662399,"""POINT (-122.45051 48.94298)""","""PUGET SOUND ENERGY INC||PUD NO…",53073010701,"""/mnt/samsung/Datasets/csv_data…",2022.111643,2021.654153
"""5YJ3E1EB6N""","""Snohomish""","""Sultan""","""WA""",98294,2022,"""TESLA""","""MODEL 3""","""Battery Electric Vehicle (BEV)""","""Eligibility unknown as battery…",0,39,185982247,"""POINT (-121.81688 47.8623)""","""PUGET SOUND ENERGY INC""",53061053803,"""/mnt/samsung/Datasets/csv_data…",2022.111643,2022.311104
"""JTJBCACB3T""","""Snohomish""","""Lake Stevens""","""WA""",98258,2026,"""LEXUS""","""RZ""","""Battery Electric Vehicle (BEV)""","""Eligibility unknown as battery…",0,39,290127582,"""POINT (-122.06402 48.01497)""","""PUGET SOUND ENERGY INC""",53061053505,"""/mnt/samsung/Datasets/csv_data…",2022.111643,2024.550751


county,city,testing_city
cat,list[cat],list[cat]
"""Benton""","[""Kennewick"", ""Richland"", … ""Richland""]","[""Kennewick"", ""Richland"", … ""Bella Vista""]"
"""Jefferson""","[""Port Townsend"", ""Port Townsend"", … ""Port Townsend""]","[""Port Townsend"", ""Nordland"", … ""Fort Drum""]"
null,"[null, null, … null]",[null]
"""Dorchester""","[""North Charleston""]","[""North Charleston""]"
"""Thurston""","[""Yelm"", ""Tumwater"", … ""Tumwater""]","[""Yelm"", ""Tumwater"", … ""Oakville""]"
…,…,…
"""Nye""","[""Pahrump""]","[""Pahrump""]"
"""Loudoun""","[""Brambleton"", ""Sterling"", ""Chantilly""]","[""Brambleton"", ""Sterling"", ""Chantilly""]"
"""Carroll""","[""Westminster""]","[""Westminster""]"


In [6]:
import polars as pl

df = pl.DataFrame({
    "dept": ["IT", "IT", "IT", "HR", "HR"],
    "dev": ["Alice", "Alice", "Bob", "Charlie", "Charlie"],
    "lang": ["Python", "Python", "Rust", "Excel", "Excel"]
})

# Using window functions (.over)
df_window = df.with_columns(
    # implode() already returns 1 list, so this works:
    all_langs = pl.col("lang").implode().over("dev"),

    # ADD .implode() after .unique() to fix the error:
    unique_langs = pl.col("lang").unique().implode().over("dept")
)

print(df_window)

shape: (5, 5)
┌──────┬─────────┬────────┬──────────────────────┬────────────────────┐
│ dept ┆ dev     ┆ lang   ┆ all_langs            ┆ unique_langs       │
│ ---  ┆ ---     ┆ ---    ┆ ---                  ┆ ---                │
│ str  ┆ str     ┆ str    ┆ list[str]            ┆ list[str]          │
╞══════╪═════════╪════════╪══════════════════════╪════════════════════╡
│ IT   ┆ Alice   ┆ Python ┆ ["Python", "Python"] ┆ ["Python", "Rust"] │
│ IT   ┆ Alice   ┆ Python ┆ ["Python", "Python"] ┆ ["Python", "Rust"] │
│ IT   ┆ Bob     ┆ Rust   ┆ ["Rust"]             ┆ ["Python", "Rust"] │
│ HR   ┆ Charlie ┆ Excel  ┆ ["Excel", "Excel"]   ┆ ["Excel"]          │
│ HR   ┆ Charlie ┆ Excel  ┆ ["Excel", "Excel"]   ┆ ["Excel"]          │
└──────┴─────────┴────────┴──────────────────────┴────────────────────┘
